In [2]:
import pandas as pd
from Bio import SeqIO

df = pd.read_csv(
    "Astroviridae_complete_1B_17nt_mmseq2.tsv",
    sep="\t",
    header=None,
    names=["center", "member"],
    dtype=str,
)

def acc(x):
    return x.split("/")[0].strip()

counts = df["center"].map(acc).value_counts().to_dict()

records = []
for rec in SeqIO.parse("Astroviridae_complete_1B_noamb_mammalia_ICTV_headers.faa", "fasta"):
    acc_id = rec.id.split("|")[0]
    if acc_id not in counts:
        continue
    n = counts[acc_id]
    new = f"{rec.description}/{n}"
    rec.description = new
    rec.id = new.split()[0]
    records.append(rec)

SeqIO.write(records, "centers.fasta", "fasta")
print(f"Записано {len(records)} центров кластеров")

Записано 130 центров кластеров


In [ ]:
import re
import pandas as pd
from Bio import SeqIO

# Роды рукокрылых, реально встречающиеся в файле (+ пара на будущее)
BAT_GENERA = {
    "Miniopterus", "Nyctalus", "Eidolon", "Rousettus",
    "Myotis", "Hipposideros", "Rhinolophus", "Scotophilus",
    "Pipistrellus", "Tylonycteris", "Mops", "Coleura",
    # на будущее, если файл обновится:
    "Pteropus", "Eptesicus", "Tadarida", "Molossus", "Carollia",
    "Artibeus", "Desmodus", "Glossophaga", "Anoura", "Megaderma",
    "Rhinopoma", "Mormopterus", "Chaerephon", "Nycteris",
    "Emballonura", "Taphozous", "Saccolaimus", "Chalinolobus",
    "Lasiurus", "Nycticeius", "Plecotus", "Barbastella", "Murina",
    "Kerivoula", "Phoniscus", "Cynopterus", "Macroglossus",
    "Eonycteris", "Epomophorus", "Epomops", "Hypsignathus",
    "Micropteropus", "Myonycteris", "Lissonycteris",
}

# Предкомпилированный регекс: \b(Genus1|Genus2|...)\b, без учёта регистра
_BAT_RE = re.compile(
    r'\b(' + '|'.join(re.escape(g) for g in sorted(BAT_GENERA, key=len, reverse=True)) + r')\b',
    re.IGNORECASE,
)
_CHIROPTERA_RE = re.compile(r'\bChiroptera\b', re.IGNORECASE)
_BAT_WORD_RE = re.compile(r'\b(bat|bats|flying[\s-]?fox|fruit[\s-]?bat)\b', re.IGNORECASE)

def is_bat(description: str) -> bool:
    """True, если запись относится к рукокрылым."""
    if _CHIROPTERA_RE.search(description):
        return True
    if _BAT_RE.search(description):
        return True
    if _BAT_WORD_RE.search(description):
        return True
    return False

# --- Основной код ---
df = pd.read_csv(
    "Astroviridae_complete_1B_17nt_mmseq2.tsv",
    sep="\t",
    header=None,
    names=["center", "member"],
    dtype=str,
)

def acc(x):
    return x.split("/")[0].strip()

counts = df["center"].map(acc).value_counts().to_dict()

records = []
skipped_bats = 0
skipped_bat_ids = []

for rec in SeqIO.parse("Astroviridae_complete_1B_noamb_mammalia_ICTV_headers.faa", "fasta"):
    acc_id = rec.id.split("|")[0]
    if acc_id not in counts:
        continue

    if is_bat(rec.description):
        skipped_bats += 1
        skipped_bat_ids.append(rec.description.split("/")[0])
        continue

    n = counts[acc_id]
    new = f"{rec.description}/{n}"
    rec.description = new
    rec.id = new.split()[0]
    records.append(rec)

SeqIO.write(records, "centers_no_bats.fasta", "fasta")
print(f"Записано {len(records)} центров (исключено {skipped_bats} летучих мышей)")

# Диагностика: какие именно акцессии были отброшены
if skipped_bat_ids:
    print("Отброшенные акцессии летучих мышей:")
    for a in sorted(set(skipped_bat_ids)):
        print("  ", a)